# <span style="color:#0f766e; font-weight:700;">Analiza Danych (Python) - Lekcja 4</span>

1. [Mini-projekt: opóźnienia lotów](#1.-Mini-projekt:-opóźnienia-lotów)
2. [Najważniejsze operacje w praktyce](#2.-Najważniejsze-operacje-w-praktyce)
3. [Dane i przygotowanie środowiska](#3.-Dane-i-przygotowanie-środowiska)
4. [Zadania obowiązkowe (1-15)](#4.-Zadania-obowiązkowe-(1-15))
5. [Zadania dodatkowe (16-24)](#5.-Zadania-dodatkowe-(16-24))
6. [Najczęstsze błędy](#Najczęstsze-błędy)


# <span style="color:#0f766e; font-weight:700;">1. Mini-projekt: opóźnienia lotów</span>

W tej lekcji pracujemy na publicznym zbiorze **nycflights13**. To duży, realistyczny zestaw danych, który dobrze domyka materiał z lekcji 1-3:
- lekcja 1: import i organizacja pracy,
- lekcja 2: statystyka opisowa i wizualizacje,
- lekcja 3: operacje na `DataFrame`, łączenie tabel i reshaping.

Tutaj dokładamy pełny pipeline analityczny:
- audyt jakości danych,
- czyszczenie braków i wartości spoza zakresu,
- bezpieczne merge wielu tabel,
- budowę cech z czasu i opóźnień,
- odpowiedzi na pytania badawcze,
- krótki raport końcowy.

## <span style="color:#0f766e; font-weight:700;">Pytania badawcze Q1-Q5</span>

- Q1: Którzy przewoźnicy mają najniższe typowe opóźnienia odlotu przy sensownej liczebności lotów?
- Q2: Które kierunki mają najwyższe średnie opóźnienia przylotu?
- Q3: Jak warunki pogodowe wiążą się z opóźnieniami odlotu?
- Q4: Jak zmienia się punktualność w czasie: miesiąc, dzień tygodnia i godzina?
- Q5: Ile rekordów odrzuciliśmy podczas czyszczenia i dlaczego?


# <span style="color:#0f766e; font-weight:700;">2. Najważniejsze operacje w praktyce</span>



Najważniejsze operacje tej lekcji:
- `query()`, `loc[]`, `iloc[]` do czytelnego wybierania rekordów,
- `assign()`, `rename()`, `sort_values()`, `set_index()` do budowy pipeline'u,
- `groupby()`, `agg()`, `value_counts()`, `nunique()` do podsumowań,
- `merge(..., validate=...)` do bezpiecznego łączenia,
- `melt()`, `pivot_table()`, `crosstab()` do zmiany kształtu danych,
- `pd.to_datetime()` i `.dt` do pracy z czasem,
- `matplotlib` i `seaborn` do szybkich wykresów roboczych.


## <span style="color:#0f766e; font-weight:700;">Przykład 1: Tabele referencyjne i szybka diagnostyka</span>

Na początek warto zobaczyć, jak wyglądają mniejsze tabele pomocnicze. Dzięki temu łatwiej później planować sensowne merge.


In [14]:
# Importujemy bibliotekę pandas do pracy z tabelami.
import pandas as pd

# Rozpoczynamy blok bezpiecznego importu danych.
try:
    # Importujemy z pakietu tylko te tabele, które będą potrzebne w tym fragmencie.
    from nycflights13 import airlines, airports, planes
# Obsługujemy sytuację, w której pakiet z danymi nie jest dostępny.
except ImportError as exc:
    # Zwracamy czytelny komunikat z instrukcją instalacji pakietu.
    raise ImportError("Zainstaluj pakiet: pip install nycflights13") from exc

# Wyświetlamy wynik bieżącego kroku, aby go od razu skontrolować.
print("airlines:", airlines.shape)

# Wyświetlamy wynik bieżącego kroku, aby go od razu skontrolować.
print(airlines.head())
# Wstawiamy pustą linię, aby oddzielić kolejne fragmenty wyniku.
print()

# Wyświetlamy wynik bieżącego kroku, aby go od razu skontrolować.
print("airports:", airports.shape)

# Wyświetlamy wynik bieżącego kroku, aby go od razu skontrolować.
print(airports[["faa", "name", "tz", "tzone"]].head())
# Wstawiamy pustą linię, aby oddzielić kolejne fragmenty wyniku.
print()

# Wyświetlamy wynik bieżącego kroku, aby go od razu skontrolować.
print("planes:", planes.shape)
# Wyświetlamy wynik bieżącego kroku, aby go od razu skontrolować.
print(planes[["tailnum", "manufacturer", "model", "seats"]].head())


airlines: (16, 2)
  carrier                    name
0      9E       Endeavor Air Inc.
1      AA  American Airlines Inc.
2      AS    Alaska Airlines Inc.
3      B6         JetBlue Airways
4      DL    Delta Air Lines Inc.

airports: (1458, 8)
   faa                           name  tz             tzone
0  04G              Lansdowne Airport  -5  America/New_York
1  06A  Moton Field Municipal Airport  -6   America/Chicago
2  06C            Schaumburg Regional  -6   America/Chicago
3  06N                Randall Airport  -5  America/New_York
4  09J          Jekyll Island Airport  -5  America/New_York

planes: (3322, 9)
  tailnum      manufacturer      model  seats
0  N10156           EMBRAER  EMB-145XR     55
1  N102UW  AIRBUS INDUSTRIE   A320-214    182
2  N103US  AIRBUS INDUSTRIE   A320-214    182
3  N104UW  AIRBUS INDUSTRIE   A320-214    182
4  N10575           EMBRAER  EMB-145LR     55


## <span style="color:#0f766e; font-weight:700;">Przykład 2: Wybieranie kolumn, filtrowanie, `query`, `loc`, `iloc`</span>

Najpierw ćwiczymy czyste operacje na fragmentach danych.


In [27]:
# Importujemy bibliotekę pandas do pracy z tabelami.
import pandas as pd

# Importujemy z pakietu tylko te tabele, które będą potrzebne w tym fragmencie.
from nycflights13 import flights

# Tworzymy tabelę `flights_day1` przez filtrowanie danych i zapisujemy wynik jako niezależną kopię.
flights_day1 = flights.query("month == 1 and day == 1").copy()

# Zapisujemy wynik bieżącej operacji do zmiennej `selected_day1`.
selected_day1 = flights_day1[["carrier", "origin", "dest", "distance", "air_time", "dep_delay"]]
# Wyświetlamy nagłówek lub komunikat: '"WYBRANE KOLUMNY:'.
print("WYBRANE KOLUMNY:")

# Wyświetlamy wynik bieżącego kroku, aby go od razu skontrolować.
print(selected_day1.head())
# Wstawiamy pustą linię, aby oddzielić kolejne fragmenty wyniku.
print()

# Zapisujemy wynik bieżącej operacji do zmiennej `long_haul_day1`.
long_haul_day1 = selected_day1.query("distance >= 1500 and air_time >= 200")
# Wyświetlamy nagłówek lub komunikat: '"LOTY DALEKIEGO ZASIĘGU:'.
print("LOTY DALEKIEGO ZASIĘGU:")
# Wyświetlamy wynik bieżącego kroku, aby go od razu skontrolować.
print(long_haul_day1.head())
# Wstawiamy pustą linię, aby oddzielić kolejne fragmenty wyniku.
print()

# Wyświetlamy nagłówek lub komunikat: '"LOC:'.
print("LOC:")
# Wyświetlamy wynik bieżącego kroku, aby go od razu skontrolować.
print(selected_day1.loc[selected_day1.index[:4], ["carrier", "dest", "distance"]])
# Wstawiamy pustą linię, aby oddzielić kolejne fragmenty wyniku.
print()

# Wyświetlamy nagłówek lub komunikat: '"ILOC:'.
print("ILOC:")
# Wyświetlamy wynik bieżącego kroku, aby go od razu skontrolować.
print(selected_day1.iloc[:4, :3])


WYBRANE KOLUMNY:
  carrier origin dest  distance  air_time  dep_delay
0      UA    EWR  IAH      1400     227.0        2.0
1      UA    LGA  IAH      1416     227.0        4.0
2      AA    JFK  MIA      1089     160.0        2.0
3      B6    JFK  BQN      1576     183.0       -1.0
4      DL    LGA  ATL       762     116.0       -6.0

LOTY DALEKIEGO ZASIĘGU:
   carrier origin dest  distance  air_time  dep_delay
12      UA    JFK  LAX      2475     345.0       -2.0
13      UA    EWR  SFO      2565     361.0       -2.0
16      UA    EWR  LAS      2227     337.0       -1.0
26      UA    JFK  SFO      2586     366.0       11.0
30      US    EWR  PHX      2133     342.0       -8.0

LOC:
  carrier dest  distance
0      UA  IAH      1400
1      UA  IAH      1416
2      AA  MIA      1089
3      B6  BQN      1576

ILOC:
  carrier origin dest
0      UA    EWR  IAH
1      UA    LGA  IAH
2      AA    JFK  MIA
3      B6    JFK  BQN


## <span style="color:#0f766e; font-weight:700;">Przykład 3: Transformacje i nowe kolumny z `sched_dep_time`</span>

Tu ćwiczymy `assign()`, `rename()` i `sort_values()` na problemie pobocznym: jak czytać godziny zapisane jako liczby `HHMM`.


In [20]:
# Importujemy bibliotekę pandas do pracy z tabelami.
import pandas as pd

# Tworzymy niezależną kopię danych i zapisujemy ją w zmiennej `schedule_demo`.
schedule_demo = flights.head(12).copy()
# Zmieniamy nazwy wybranych kolumn i zapisujemy wynik do `schedule_demo`.
schedule_demo = schedule_demo.rename(columns={"distance": "distance_miles"})
# Tworzymy lub nadpisujemy kolumnę `sched_dep_hour` w tabeli `schedule_demo`.
schedule_demo["sched_dep_hour"] = schedule_demo["sched_dep_time"] // 100
# Tworzymy lub nadpisujemy kolumnę `sched_dep_minute` w tabeli `schedule_demo`.
schedule_demo["sched_dep_minute"] = schedule_demo["sched_dep_time"] % 100
# Tworzymy lub nadpisujemy kolumnę `sched_arr_hour` w tabeli `schedule_demo`.
schedule_demo["sched_arr_hour"] = schedule_demo["sched_arr_time"] // 100
# Tworzymy lub nadpisujemy kolumnę `sched_arr_minute` w tabeli `schedule_demo`.
schedule_demo["sched_arr_minute"] = schedule_demo["sched_arr_time"] % 100
# Tworzymy lub nadpisujemy kolumnę `distance_km` w tabeli `schedule_demo`.
schedule_demo["distance_km"] = (schedule_demo["distance_miles"] * 1.609).round(1)
# Łączymy dwa kody lotnisk w jeden napis i zapisujemy go w kolumnie `route`.
schedule_demo["route"] = schedule_demo["origin"] + "-" + schedule_demo["dest"]
# Porządkujemy obiekt `schedule_demo` zgodnie z podaną regułą sortowania.
schedule_demo = schedule_demo.sort_values(["sched_dep_hour", "sched_dep_minute"])
# Wyświetlamy wynik bieżącego kroku, aby go od razu skontrolować.
print(schedule_demo[["route", "sched_dep_time", "sched_dep_hour", "sched_dep_minute", "distance_km"]])


      route  sched_dep_time  sched_dep_hour  sched_dep_minute  distance_km
0   EWR-IAH             515               5                15       2252.6
1   LGA-IAH             529               5                29       2278.3
2   JFK-MIA             540               5                40       1752.2
3   JFK-BQN             545               5                45       2535.8
5   EWR-ORD             558               5                58       1156.9
4   LGA-ATL             600               6                 0       1226.1
6   EWR-FLL             600               6                 0       1713.6
7   LGA-IAD             600               6                 0        368.5
8   JFK-MCO             600               6                 0       1518.9
9   LGA-ORD             600               6                 0       1179.4
10  JFK-PBI             600               6                 0       1654.1
11  JFK-TPA             600               6                 0       1617.0


## <span style="color:#0f766e; font-weight:700;">Przykład 4: `groupby`, `agg`, `nunique` na pytaniu pobocznym</span>

Zamiast analizować ogólne opóźnienia, policzymy ile czasu loty potrafiły „odrobić” w powietrzu.


In [51]:
# Importujemy bibliotekę pandas do pracy z tabelami.
import pandas as pd

# Tworzymy tabelę `recovery_demo` po usunięciu rekordów z brakami w wybranych kolumnach.
recovery_demo = flights.dropna(subset=["dep_delay", "arr_delay", "carrier", "origin"]).copy()
# Obliczamy i zapisujemy w kolumnie `delay_recovery` różnicę między dwiema zmiennymi.
recovery_demo["delay_recovery"] = recovery_demo["dep_delay"] - recovery_demo["arr_delay"]
# Budujemy tabelę `recovery_summary` z agregacjami policzonymi dla każdej grupy.
recovery_summary = recovery_demo.groupby(["carrier", "origin"]).agg(
    # Zapisujemy wynik bieżącej operacji do zmiennej `flights_n`.
    flights_n=("flight", "count"),
    # Zapisujemy wynik bieżącej operacji do zmiennej `routes_n`.
    routes_n=("dest", "nunique"),
    # Zapisujemy wynik bieżącej operacji do zmiennej `recovery_mean`.
    recovery_mean=("delay_recovery", "mean"),
    # Zapisujemy wynik bieżącej operacji do zmiennej `recovery_median`.
    recovery_median=("delay_recovery", "median"),
# Domykamy bieżące wyrażenie lub blok parametrów.
)
# Wyświetlamy wynik bieżącego kroku, aby go od razu skontrolować.
print(recovery_summary.sort_values("recovery_mean", ascending=False).head(12).round(2))


                flights_n  routes_n  recovery_mean  recovery_median
carrier origin                                                     
AS      EWR           709         1          15.76             17.0
VX      EWR          1552         2          12.62             14.0
HA      JFK           342         1          11.82             11.0
DL      JFK         20559        29          10.67             13.0
VX      JFK          3564         5          10.28             12.0
9E      JFK         13742        34           9.86             12.0
EV      LGA          8225        45           9.71             12.0
WN      LGA          5988         8           9.30             11.0
AA      EWR          3363         3           9.01             11.0
UA      EWR         45501        47           8.95             11.0
AA      JFK         13600        17           8.19             10.0
        LGA         14984         5           8.03             10.0


## <span style="color:#0f766e; font-weight:700;">Przykład 5: `merge()` z tabelą `planes`</span>

Ćwiczymy merge na innym pytaniu niż w projekcie: jak wygląda rozkład liczby miejsc w samolotach używanych przez wybranych przewoźników.


In [ ]:
# Importujemy bibliotekę pandas do pracy z tabelami.
import pandas as pd

# Tworzymy tabelę `planes_demo` przez filtrowanie danych i zapisujemy wynik jako niezależną kopię.
planes_demo = flights.query("carrier in ['AA', 'DL', 'UA']").head(2000).copy()

# Tworzymy niezależną kopię danych i zapisujemy ją w zmiennej `planes_lookup`.
planes_lookup = planes[["tailnum", "manufacturer", "model", "seats", "engine"]].copy()

# Tworzymy obiekt `planes_demo` przez połączenie dwóch tabel na wspólnym kluczu.
planes_demo = planes_demo.merge(
    # Przekazujemy obiekt `planes_lookup` jako argument do bieżącej funkcji.
    planes_lookup,
    # Parametr `on` wskazuje kolumnę lub listę kolumn używanych jako klucz łączenia.
    on="tailnum",
    # Parametr `how` określa typ połączenia tabel i to, które rekordy zostają zachowane.
    how="left",
    # Parametr `validate` sprawdza, czy relacja między kluczami ma oczekiwaną strukturę.
    validate="many_to_one",
    # Parametr `indicator` dodaje kolumnę z informacją o źródle dopasowania po merge.
    indicator="merge_planes",
# Domykamy bieżące wyrażenie lub blok parametrów.
)

# Wyświetlamy wynik bieżącego kroku, aby go od razu skontrolować.
print(planes_demo["merge_planes"].value_counts())
# Wstawiamy pustą linię, aby oddzielić kolejne fragmenty wyniku.
print()

# Wyświetlamy wynik bieżącego kroku, aby go od razu skontrolować.
print(planes_demo[["carrier", "tailnum", "manufacturer", "model", "seats", "merge_planes"]].head(12))


## <span style="color:#0f766e; font-weight:700;">Przykład 6: `melt()` i `pivot_table()` na podsumowaniu miesięcznym</span>

To jest dobry moment, żeby przypomnieć zmianę kształtu danych z lekcji 3.


In [ ]:
# Importujemy bibliotekę pandas do pracy z tabelami.
import pandas as pd

# Budujemy tabelę `monthly_origin_demo` z agregacjami policzonymi dla każdej grupy.
monthly_origin_demo = flights.dropna(subset=["dep_delay", "arr_delay"]).groupby(["month", "origin"]).agg(
    # Zapisujemy wynik bieżącej operacji do zmiennej `dep_delay_mean`.
    dep_delay_mean=("dep_delay", "mean"),
    # Zapisujemy wynik bieżącej operacji do zmiennej `arr_delay_mean`.
    arr_delay_mean=("arr_delay", "mean"),
# Zamykamy agregację i zamieniamy indeks grupowania z powrotem na zwykłe kolumny.
).reset_index()

# Tworzymy tabelę przestawną i zapisujemy ją jako `monthly_origin_pivot`.
monthly_origin_pivot = monthly_origin_demo.pivot_table(
    # Parametr `index` wskazuje, co ma trafić do wierszy wyniku.
    index="month",
    # Parametr `columns` ustawia kolumny w wyniku albo słownik zmian nazw kolumn.
    columns="origin",
    # Parametr `values` wybiera zmienną, która ma być agregowana lub pokazana w tabeli.
    values="dep_delay_mean",
    # Parametr `aggfunc` określa funkcję agregującą używaną przy budowie tabeli.
    aggfunc="mean",
# Domykamy bieżące wyrażenie lub blok parametrów.
)

# Zapisujemy wynik bieżącej operacji do zmiennej `monthly_origin_long`.
monthly_origin_long = monthly_origin_pivot.reset_index().melt(
    # Parametr `id_vars` wskazuje kolumny, które mają pozostać identyfikatorami po stopieniu tabeli.
    id_vars="month",
    # Parametr `var_name` ustawia nazwę kolumny z dawnymi nagłówkami po operacji `melt`.
    var_name="origin",
    # Parametr `value_name` ustawia nazwę kolumny z wartościami po operacji `melt`.
    value_name="dep_delay_mean",
# Domykamy bieżące wyrażenie lub blok parametrów.
)

# Wyświetlamy nagłówek lub komunikat: '"PIVOT:'.
print("PIVOT:")
# Wyświetlamy wynik bieżącego kroku, aby go od razu skontrolować.
print(monthly_origin_pivot.head())
# Wstawiamy pustą linię, aby oddzielić kolejne fragmenty wyniku.
print()
# Wyświetlamy nagłówek lub komunikat: '"MELT:'.
print("MELT:")
# Wyświetlamy wynik bieżącego kroku, aby go od razu skontrolować.
print(monthly_origin_long.head(12))


## <span style="color:#0f766e; font-weight:700;">Przykład 7: `pd.to_datetime()`, `.dt` i `crosstab()`</span>

Na tym przykładzie ćwiczymy pracę z czasem oraz tabelę przekrojową.


In [ ]:
# Importujemy bibliotekę pandas do pracy z tabelami.
import pandas as pd

# Tworzymy niezależną kopię danych i zapisujemy ją w zmiennej `time_demo`.
time_demo = flights.head(5000).copy()
# Nadpisujemy kolumnę `time_hour` w tabeli `time_demo` po konwersji do typu datetime.
time_demo["time_hour"] = pd.to_datetime(time_demo["time_hour"], utc=True)
# Zapisujemy do kolumny `quarter` cechę czasu wyliczoną z daty.
time_demo["quarter"] = time_demo["time_hour"].dt.quarter
# Zapisujemy do kolumny `weekday` cechę czasu wyliczoną z daty.
time_demo["weekday"] = time_demo["time_hour"].dt.day_name()
# Tworzymy w kolumnie `is_weekend` flagę na podstawie sprawdzenia przynależności do listy.
time_demo["is_weekend"] = time_demo["weekday"].isin(["Saturday", "Sunday"])
# Tworzymy tabelę krzyżową i zapisujemy wynik do `weekend_table`.
weekend_table = pd.crosstab(
    # Przekazujemy jedną z kolumn wejściowych do tabeli krzyżowej.
    time_demo["quarter"],
    # Przekazujemy jedną z kolumn wejściowych do tabeli krzyżowej.
    time_demo["is_weekend"],
    # Parametr `normalize` zamienia liczebności na udziały według wskazanego kierunku normalizacji.
    normalize="index",
# Domykamy bieżące wyrażenie lub blok parametrów.
)
# Wyświetlamy wynik bieżącego kroku, aby go od razu skontrolować.
print(weekend_table)


## <span style="color:#0f766e; font-weight:700;">Przykład 8: Dwa wykresy robocze</span>




In [ ]:
# Importujemy moduł `pyplot` do budowy wykresów.
import matplotlib.pyplot as plt
# Importujemy bibliotekę seaborn do estetycznych wizualizacji.
import seaborn as sns

# Tworzymy tabelę `plot_demo` po usunięciu rekordów z brakami w wybranych kolumnach.
plot_demo = flights.dropna(subset=["distance", "air_time", "carrier", "origin"]).copy()

# Zapisujemy wynik bieżącej operacji do zmiennej `plot_demo`.
plot_demo = plot_demo.query("distance <= 3000 and air_time <= 450")

# Tworzymy nowe płótno wykresu z podanym rozmiarem.
plt.figure(figsize=(8, 4))

# Rysujemy wykres rozrzutu dla wskazanych zmiennych.
sns.scatterplot(
    # Parametr `data` wskazuje tabelę, z której funkcja ma pobrać dane.
    data=plot_demo.sample(min(2000, len(plot_demo)), random_state=42),
    # Parametr `x` wskazuje zmienną odkładaną na osi poziomej.
    x="distance",
    # Parametr `y` wskazuje zmienną odkładaną na osi pionowej.
    y="air_time",
    # Parametr `alpha` steruje przezroczystością elementów na wykresie.
    alpha=0.25,
# Domykamy bieżące wyrażenie lub blok parametrów.
)
# Ustawiamy tytuł wykresu.
plt.title("distance vs air_time")
# Ustawiamy podpis osi poziomej.
plt.xlabel("distance")
# Ustawiamy podpis osi pionowej.
plt.ylabel("air_time")
# Dopasowujemy marginesy, aby elementy wykresu się nie nakładały.
plt.tight_layout()
# Wyświetlamy gotowy wykres.
plt.show()

# Tworzymy nowe płótno wykresu z podanym rozmiarem.
plt.figure(figsize=(8, 4))

# Rysujemy wykres pudełkowy porównujący rozkład między kategoriami.
sns.boxplot(data=plot_demo[plot_demo["origin"].isin(["EWR", "JFK", "LGA"])], x="origin", y="air_time")
# Ustawiamy tytuł wykresu.
plt.title("air_time wg origin")
# Ustawiamy podpis osi poziomej.
plt.xlabel("origin")
# Ustawiamy podpis osi pionowej.
plt.ylabel("air_time")
# Dopasowujemy marginesy, aby elementy wykresu się nie nakładały.
plt.tight_layout()
# Wyświetlamy gotowy wykres.
plt.show()


# <span style="color:#0f766e; font-weight:700;">3. Dane i przygotowanie środowiska</span>

W tej lekcji pracujemy na pakiecie:
- `nycflights13`

Instalacja:
- `pip install nycflights13`

Główne tabele:
- `flights`
- `airlines`
- `airports`
- `weather`
- `planes`

Źródła:
- https://pypi.org/project/nycflights13/
- https://github.com/tidyverse/nycflights13


# <span style="color:#0f766e; font-weight:700;">4. Zadania obowiązkowe (1-15)</span>


## <span style="color:#0f766e; font-weight:700;">Zadanie 1</span>

Wczytaj wszystkie tabele z `nycflights13` i wykonaj pełną szybką diagnostykę.

Pracuj na tabelach:
- `flights`
- `airlines`
- `airports`
- `weather`
- `planes`

Dla każdej z nich pokaż:
- `shape`
- listę kolumn przez `columns`
- pierwsze 5 rekordów przez `head()`
- strukturę przez `info()`

Najlepiej zrób to w pętli po słowniku `{'flights': flights, ...}`.

Wymagane funkcje/parametry:
- `from nycflights13 import ...` - import tabel z pakietu.
- `shape`, `columns`, `head()`, `info()` - szybka diagnostyka.

Kryterium zaliczenia:
- zadanie wykonane zgodnie z poleceniem,
- wynik pokazany w notebooku.

Checkpoint:
- pokaż prowadzącemu wynik końcowy tej sekcji.

Dokumentacja:
- https://pypi.org/project/nycflights13/
- https://pandas.pydata.org/docs/reference/frame.html


In [139]:
# tutaj dodaj swój kod
import pandas as pd
from nycflights13 import flights, airlines, airports, weather, planes
datasets = {'flights': flights, 
            'airlines': airlines,
            'airports': airports,
            'weather': weather, 
            'planes': planes}
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print("-"*40)
    print(f'{name}:', df.shape)
    print(df.head())
    print(df.columns)
    print(df.info())
    



FLIGHTS
----------------------------------------
flights: (336776, 19)
   year  month  day  dep_time  sched_dep_time  dep_delay  arr_time  \
0  2013      1    1     517.0             515        2.0     830.0   
1  2013      1    1     533.0             529        4.0     850.0   
2  2013      1    1     542.0             540        2.0     923.0   
3  2013      1    1     544.0             545       -1.0    1004.0   
4  2013      1    1     554.0             600       -6.0     812.0   

   sched_arr_time  arr_delay carrier  flight tailnum origin dest  air_time  \
0             819       11.0      UA    1545  N14228    EWR  IAH     227.0   
1             830       20.0      UA    1714  N24211    LGA  IAH     227.0   
2             850       33.0      AA    1141  N619AA    JFK  MIA     160.0   
3            1022      -18.0      B6     725  N804JB    JFK  BQN     183.0   
4             837      -25.0      DL     461  N668DN    LGA  ATL     116.0   

   distance  hour  minute             

## <span style="color:#0f766e; font-weight:700;">Zadanie 2</span>

Przygotuj audyt jakości dla `flights` i `weather`.

Dla obu tabel zbuduj osobną tabelę audytu z dwiema kolumnami:
- `missing_n`
- `missing_pct`

Następnie:
- w `flights` policz duplikaty pełnych rekordów,
- w `weather` policz duplikaty po kluczu `origin`, `year`, `month`, `day`, `hour`.

Na końcu pokaż:
- `head(12)` dla tabeli braków `flights`,
- `head(12)` dla tabeli braków `weather`,
- liczbę duplikatów w obu przypadkach.

Wymagane funkcje/parametry:
- `isna().sum()` i `isna().mean()` - podsumowanie braków.
- `duplicated()` - wykrywanie duplikatów.
- `sort_values()` - ranking kolumn z największym problemem.

Kryterium zaliczenia:
- zadanie wykonane zgodnie z poleceniem,
- wynik pokazany w notebooku.

Checkpoint:
- pokaż prowadzącemu wynik końcowy tej sekcji.

Dokumentacja:
- https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html


In [127]:
# tutaj dodaj swój kod

# audyt braków dla flights
flights_missing = pd.DataFrame({
    "missing_n": flights.isna().sum(),
    "missing_pct": flights.isna().mean()*100
}).sort_values("missing_n", ascending=False) # asceding=False - sortowanie malejąco


# audyt braków dla weather
weather_missing = pd.DataFrame({
    "missing_n": weather.isna().sum(),
    "missing_pct": weather.isna().mean()*100
}).sort_values("missing_n", ascending=False)

flights_duplicates = flights.duplicated().sum()

weather_duplicates = weather.duplicated(subset=["origin", "year", "month", "day", "hour"]).sum()


print("Flights - missing values:")
print(flights_missing.head(12))

print("\nWeather - missing values:")
print(weather_missing.head(12))

print("\nFlights duplicates:", flights_duplicates)
print("Weather duplicates:", weather_duplicates)


Flights - missing values:
           missing_n  missing_pct
arr_delay       9430     2.800081
air_time        9430     2.800081
arr_time        8713     2.587180
dep_time        8255     2.451184
dep_delay       8255     2.451184
tailnum         2512     0.745896
year               0     0.000000
origin             0     0.000000
minute             0     0.000000
hour               0     0.000000
distance           0     0.000000
dest               0     0.000000

Weather - missing values:
            missing_n  missing_pct
wind_gust       20778    79.563469
pressure         2729    10.449933
wind_dir          460     1.761440
wind_speed          4     0.015317
temp                1     0.003829
dewp                1     0.003829
humid               1     0.003829
origin              0     0.000000
year                0     0.000000
month               0     0.000000
day                 0     0.000000
hour                0     0.000000

Flights duplicates: 0
Weather duplicates: 3


## <span style="color:#0f766e; font-weight:700;">Zadanie 3</span>

Przygotuj roboczą tabelę `project_flights` tylko z kolumnami potrzebnymi do projektu.

Wybierz z `flights` dokładnie te kolumny:
- `year`
- `month`
- `day`
- `dep_time`
- `sched_dep_time`
- `dep_delay`
- `arr_time`
- `sched_arr_time`
- `arr_delay`
- `carrier`
- `flight`
- `tailnum`
- `origin`
- `dest`
- `air_time`
- `distance`
- `hour`
- `minute`
- `time_hour`

Zapisz wynik pod nazwą `project_flights`.

To jest zestaw potrzebny później do:
- czyszczenia braków,
- walidacji zakresów,
- budowy nowych cech,
- łączenia z pogodą, przewoźnikami i lotniskami.

Następnie pokaż 10 rekordów o największym `distance` przez `query()` lub `sort_values()`.

Wymagane funkcje/parametry:
- wybór kolumn `df[[...]]` - budowa węższej tabeli roboczej.
- `query()` albo `sort_values()` - podgląd skrajnych rekordów.
- `copy()` - bezpieczna kopia danych.

Kryterium zaliczenia:
- zadanie wykonane zgodnie z poleceniem,
- wynik pokazany w notebooku.

Checkpoint:
- pokaż prowadzącemu wynik końcowy tej sekcji.

Dokumentacja:
- https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.query.html
- https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html


In [129]:
project_flights = flights[["year", "month", "day", "dep_time", "sched_dep_time", "dep_delay", "arr_time", "sched_arr_time", "arr_delay", "carrier", "flight", 'tailnum', "origin", "dest", "air_time", "distance", 'hour', 'minute', 'time_hour']].copy()
project_flights.sort_values("distance", ascending=False).head(10)

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
50676,2013,10,26,1004.0,1000,4.0,1435.0,1450,-15.0,HA,51,N386HA,JFK,HNL,608.0,4983,10,0,2013-10-26T14:00:00Z
108078,2013,12,28,933.0,930,3.0,1520.0,1535,-15.0,HA,51,N384HA,JFK,HNL,633.0,4983,9,30,2013-12-28T14:00:00Z
100067,2013,12,19,924.0,930,-6.0,1450.0,1535,-45.0,HA,51,N386HA,JFK,HNL,609.0,4983,9,30,2013-12-19T14:00:00Z
179566,2013,4,16,953.0,1000,-7.0,1443.0,1510,-27.0,HA,51,N381HA,JFK,HNL,631.0,4983,10,0,2013-04-16T14:00:00Z
30229,2013,10,4,954.0,1000,-6.0,1438.0,1450,-12.0,HA,51,N380HA,JFK,HNL,618.0,4983,10,0,2013-10-04T14:00:00Z
328582,2013,9,22,952.0,1000,-8.0,1439.0,1445,-6.0,HA,51,N386HA,JFK,HNL,626.0,4983,10,0,2013-09-22T14:00:00Z
9947,2013,1,12,901.0,900,1.0,1452.0,1530,-38.0,HA,51,N383HA,JFK,HNL,615.0,4983,9,0,2013-01-12T14:00:00Z
269447,2013,7,21,955.0,1000,-5.0,1448.0,1430,18.0,HA,51,N383HA,JFK,HNL,629.0,4983,10,0,2013-07-21T14:00:00Z
141043,2013,3,6,855.0,900,-5.0,1459.0,1540,-41.0,HA,51,N385HA,JFK,HNL,640.0,4983,9,0,2013-03-06T14:00:00Z
18433,2013,1,22,853.0,900,-7.0,1506.0,1530,-24.0,HA,51,N381HA,JFK,HNL,632.0,4983,9,0,2013-01-22T14:00:00Z


## <span style="color:#0f766e; font-weight:700;">Zadanie 4</span>

Zbuduj `flights_no_missing` przez usunięcie braków w kluczowych kolumnach.

Pracuj na `project_flights` z zadania 3.

Usuń rekordy z brakami w dokładnie tych kolumnach:
- `dep_delay`
- `arr_delay`
- `carrier`
- `dest`
- `time_hour`
- `air_time`
- `distance`

Zapisz wynik pod nazwą `flights_no_missing`.

Pokaż:
- rozmiar `project_flights`,
- rozmiar `flights_no_missing`,
- liczbę usuniętych rekordów.

Wymagane funkcje/parametry:
- `dropna(subset=[...])` - usuwanie braków w wybranych kolumnach.
- `shape` - porównanie rozmiaru tabeli przed i po czyszczeniu.

Kryterium zaliczenia:
- zadanie wykonane zgodnie z poleceniem,
- wynik pokazany w notebooku.

Checkpoint:
- pokaż prowadzącemu wynik końcowy tej sekcji.

Dokumentacja:
- https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html


In [130]:

flights_no_missing = project_flights.dropna(subset=["dep_delay", "arr_delay", "carrier", 'dest', 'time_hour', 'air_time', 'distance']).copy()

print('rozmiar project_flights:', project_flights.shape)
print('rozmiar flights_no_missing:', flights_no_missing.shape)

removed_rows = project_flights.shape[0] - flights_no_missing.shape[0] # shape[0] - liczba wierszy, shape[1] - liczba kolumn
print('liczba usuniętych wierszy:', removed_rows)


rozmiar project_flights: (336776, 19)
rozmiar flights_no_missing: (327346, 19)
liczba usuniętych wierszy: 9430


## <span style="color:#0f766e; font-weight:700;">Zadanie 5</span>

Zastosuj reguły jakości zakresów i utwórz `flights_clean`.

Pracuj na `flights_no_missing` z zadania 4.

Na potrzeby tej lekcji przyjmij dokładnie następujące zakresy:
- `dep_delay` od `-60` do `360`
- `arr_delay` od `-100` do `360`
- `air_time` od `20` do `500`
- `distance` od `80` do `5000`

Traktujemy je jako robocze reguły jakości na zajęciach:
- bardzo wczesne odloty poniżej `-60` minut odrzucamy,
- skrajnie duże opóźnienia powyżej `360` minut odrzucamy,
- loty z `air_time < 20` uznajemy za problematyczne,
- dystanse poza zakresem `80-5000` traktujemy jako niespójne dla tej analizy.

Zapisz wynik pod nazwą `flights_clean`.

Pokaż:
- rozmiar `flights_no_missing`,
- rozmiar `flights_clean`,
- liczbę rekordów odrzuconych przez walidację zakresów.

Wymagane funkcje/parametry:
- `between()` - kontrola zakresów liczbowych.
- maski logiczne `&` - łączenie reguł.
- `df[mask]` - filtrowanie poprawnych rekordów.

Kryterium zaliczenia:
- zadanie wykonane zgodnie z poleceniem,
- wynik pokazany w notebooku.

Checkpoint:
- pokaż prowadzącemu wynik końcowy tej sekcji.

Dokumentacja:
- https://pandas.pydata.org/docs/reference/api/pandas.Series.between.html


In [131]:
# tutaj dodaj swój kod

mask = (
    flights_no_missing["dep_delay"].between(-60, 360) &
    flights_no_missing["arr_delay"].between(-100, 360) &
    flights_no_missing["air_time"].between(20, 500) &
    flights_no_missing["distance"].between(80, 5000)
)
flights_clean = flights_no_missing[mask]

print('rozmiar flights_no_missing:', flights_no_missing.shape)
print('rozmiar flights_clean:', flights_clean.shape)    

removed = flights_no_missing.shape[0] - flights_clean.shape[0]
print('liczba rekordów odrzuconych przez walidację zakresów:', removed)

rozmiar flights_no_missing: (327346, 19)
rozmiar flights_clean: (326369, 19)
liczba rekordów odrzuconych przez walidację zakresów: 977


## <span style="color:#0f766e; font-weight:700;">Zadanie 6</span>

Przygotuj `weather_clean` do późniejszego merge z lotami.

Pracuj na tabeli `weather`.

Wybierz dokładnie te kolumny:
- `origin`
- `year`
- `month`
- `day`
- `hour`
- `temp`
- `wind_speed`
- `precip`
- `visib`
- `pressure`
- `time_hour`

Następnie:
1. posortuj tabelę po `time_hour`,
2. usuń duplikaty po kluczu `origin`, `year`, `month`, `day`, `hour`,
3. zapisz wynik pod nazwą `weather_clean`.

Na końcu pokaż:
- rozmiar `weather_clean`,
- liczbę duplikatów po kluczu po czyszczeniu.

Wymagane funkcje/parametry:
- wybór kolumn `df[[...]]` - zawężenie danych pogodowych.
- `sort_values()` - uporządkowanie wierszy przed deduplikacją.
- `drop_duplicates(subset=[...])` - ujednoznacznienie klucza merge.

Kryterium zaliczenia:
- zadanie wykonane zgodnie z poleceniem,
- wynik pokazany w notebooku.

Checkpoint:
- pokaż prowadzącemu wynik końcowy tej sekcji.

Dokumentacja:
- https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html


In [132]:
# tutaj dodaj swój kod

weather_clean = weather[['origin', 'year', 'month', 'day', 'hour', 'temp', 'wind_speed', 'precip', 'visib', 'pressure', 'time_hour']].copy()
weather_clean.sort_values('time_hour').head(15)

weather_clean = weather_clean.drop_duplicates(
    subset=['origin', 'year', 'month', 'day', 'hour'])

print('rozmiar weather clean:', weather_clean.shape)

print('liczba duplikatów po czyszczeniu:', int(
    weather_clean.duplicated(subset=['origin', 'year', 'month', 'day', 'hour']).sum()))

rozmiar weather clean: (26112, 11)
liczba duplikatów po czyszczeniu: 0


## <span style="color:#0f766e; font-weight:700;">Zadanie 7</span>

Dodaj cechy analityczne i zapisz wynik jako `flights_feat`.

Pracuj na `flights_clean` z zadania 5.

Wykonaj kolejno:
1. przekonwertuj `time_hour` do typu datetime,
2. dodaj `is_delayed_dep = 1`, gdy `dep_delay > 15`, w przeciwnym razie `0`,
3. dodaj `is_delayed_arr = 1`, gdy `arr_delay > 15`, w przeciwnym razie `0`,
4. dodaj `weekday` z `time_hour`,
5. dodaj `month_num` z `time_hour`,
6. dodaj `dep_hour` z `time_hour`,
7. dodaj `delay_recovery = dep_delay - arr_delay`,
8. dodaj `dep_delay_class` przez `pd.cut()` z klasami:
   - `early`
   - `on_time`
   - `delay_16_60`
   - `delay_60_plus`

Użyj tych samych granic klas:
- `[-100, -1, 15, 60, 360]`

Zapisz wynik pod nazwą `flights_feat`.

Wymagane funkcje/parametry:
- `pd.to_datetime()` - konwersja kolumny czasu.
- `assign()` - dodanie kilku kolumn naraz.
- `.dt.day_name()` i `.dt.hour` - cechy z czasu.
- `pd.cut()` - klasy opóźnień.

Kryterium zaliczenia:
- zadanie wykonane zgodnie z poleceniem,
- wynik pokazany w notebooku.

Checkpoint:
- pokaż prowadzącemu wynik końcowy tej sekcji.

Dokumentacja:
- https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html
- https://pandas.pydata.org/docs/reference/api/pandas.cut.html


In [133]:
# tutaj dodaj swój kod

flights_clean['time_hour'] = pd.to_datetime(flights_clean['time_hour'])

flights_feat = flights_clean.assign(
    is_delayed_dep=(flights_clean["dep_delay"] > 15).astype(int),
    is_delayed_arr=(flights_clean["arr_delay"] > 15).astype(int),
    weekday=flights_clean["time_hour"].dt.day_name(), #wyciąga dzień tygodnia z daty Monday itd.
    month_num=flights_clean["time_hour"].dt.month, #wyciąga numer miesiąca z daty 1-12
    dep_hour=flights_clean["time_hour"].dt.hour, #wyciąga godzinę z daty 0-23
    delay_recovery=flights_clean["dep_delay"] - flights_clean["arr_delay"] #pokazuje ile opóźnienia zostało "odrobione" w trakcie lotu, czyli ile minut opóźnienia odlotu zostało zrekompensowane przez wcześniejszy przylot
)

flights_feat["dep_delay_class"] = pd.cut(
    flights_feat["dep_delay"],
    bins=[-100, -1, 15, 60, 360],
    labels=["early", "on_time", "delay_16_60", "delay_60_plus"]
)
flights_feat


,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,...,hour,minute,time_hour,is_delayed_dep,is_delayed_arr,weekday,month_num,dep_hour,delay_recovery,dep_delay_class
0,2013,1,1,517.0,515,2.0,830.0,819,11.0,UA,...,5,15,2013-01-01 10:00:00+00:00,0,0,Tuesday,1,10,-9.0,on_time
1,2013,1,1,533.0,529,4.0,850.0,830,20.0,UA,...,5,29,2013-01-01 10:00:00+00:00,0,1,Tuesday,1,10,-16.0,on_time
2,2013,1,1,542.0,540,2.0,923.0,850,33.0,AA,...,5,40,2013-01-01 10:00:00+00:00,0,1,Tuesday,1,10,-31.0,on_time
3,2013,1,1,544.0,545,-1.0,1004.0,1022,-18.0,B6,...,5,45,2013-01-01 10:00:00+00:00,0,0,Tuesday,1,10,17.0,early
4,2013,1,1,554.0,600,-6.0,812.0,837,-25.0,DL,...,6,0,2013-01-01 11:00:00+00:00,0,0,Tuesday,1,11,19.0,early
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
336765,2013,9,30,2240.0,2245,-5.0,2334.0,2351,-17.0,B6,...,22,45,2013-10-01 02:00:00+00:00,0,0,Tuesday,10,2,12.0,early
336766,2013,9,30,2240.0,2250,-10.0,2347.0,7,-20.0,B6,...,22,50,2013-10-01 02:00:00+00:00,0,0,Tuesday,10,2,10.0,early
336767,2013,9,30,2241.0,2246,-5.0,2345.0,1,-16.0,B6,...,22,46,2013-10-01 02:00:00+00:00,0,0,Tuesday,10,2,11.0,early
336768,2013,9,30,2307.0,2255,12.0,2359.0,2358,1.0,B6,...,22,55,2013-10-01 02:00:00+00:00,0,0,Tuesday,10,2,11.0,on_time


## <span style="color:#0f766e; font-weight:700;">Zadanie 8</span>

Połącz `flights_feat` z `airlines` i `airports`.

Pracuj tak:
1. do `flights_feat` dołącz `airlines` po kolumnie `carrier`,
2. z `airports` przygotuj lookup tylko dla lotnisk docelowych z kolumn:
   - `faa`
   - `name`
   - `lat`
   - `lon`
3. zmień nazwy tak, aby po merge mieć:
   - `dest`
   - `dest_airport_name`
   - `dest_lat`
   - `dest_lon`
4. dołącz tę tabelę do lotów po `dest`.

W obu merge:
- użyj `validate='many_to_one'`,
- dodaj `indicator`,
- końcową tabelę zapisz jako `flights_geo`.

Na końcu pokaż:
- `value_counts()` dla statusu merge z `airlines`,
- `value_counts()` dla statusu merge z `airports`,
- kilka pierwszych wierszy z nowymi kolumnami.

Wymagane funkcje/parametry:
- `merge()` - łączenie tabel.
- `rename(columns={...})` - czytelne nazwy po merge z airports.
- `validate='many_to_one'` i `indicator` - kontrola merge.

Kryterium zaliczenia:
- zadanie wykonane zgodnie z poleceniem,
- wynik pokazany w notebooku.

Checkpoint:
- pokaż prowadzącemu wynik końcowy tej sekcji.

Dokumentacja:
- https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html
- https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rename.html


In [134]:
# tutaj dodaj swój kod

# merge z airlines
flights_air = flights_feat.merge(
    airlines,
    on="carrier",
    how="left",
    validate="many_to_one",
    indicator="airlines_merge"
)

airports_dest = airports[["faa", "name", "lat", "lon"]]

airports_dest = airports_dest.rename(columns={
    "faa": "dest",
    "name": "dest_airport_name",
    "lat": "dest_lat",
    "lon": "dest_lon"
})

# merge z airports
flights_geo = flights_air.merge(
    airports_dest,
    on="dest",
    how="left",
    validate="many_to_one",
    indicator="airports_merge"
)


print(flights_geo["airlines_merge"].value_counts())
print('\n') #zeby oddzielić wyniki dwóch merge'y
print(flights_geo["airports_merge"].value_counts())


flights_geo.head()

airlines_merge
both          326369
left_only          0
right_only         0
Name: count, dtype: int64


airports_merge
both          318834
left_only       7535
right_only         0
Name: count, dtype: int64


,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,...,month_num,dep_hour,delay_recovery,dep_delay_class,name,airlines_merge,dest_airport_name,dest_lat,dest_lon,airports_merge
0,2013,1,1,517.0,515,2.0,830.0,819,11.0,UA,...,1,10,-9.0,on_time,United Air Lines Inc.,both,George Bush Intercontinental,29.984433,-95.341442,both
1,2013,1,1,533.0,529,4.0,850.0,830,20.0,UA,...,1,10,-16.0,on_time,United Air Lines Inc.,both,George Bush Intercontinental,29.984433,-95.341442,both
2,2013,1,1,542.0,540,2.0,923.0,850,33.0,AA,...,1,10,-31.0,on_time,American Airlines Inc.,both,Miami Intl,25.793250,-80.290556,both
3,2013,1,1,544.0,545,-1.0,1004.0,1022,-18.0,B6,...,1,10,17.0,early,JetBlue Airways,both,NaN,NaN,NaN,left_only
4,2013,1,1,554.0,600,-6.0,812.0,837,-25.0,DL,...,1,11,19.0,early,Delta Air Lines Inc.,both,Hartsfield Jackson Atlanta Intl,33.636719,-84.428067,both


## <span style="color:#0f766e; font-weight:700;">Zadanie 9</span>

Połącz `flights_geo` z `weather_clean` i utwórz `analysis_df`.

Pracuj na:
- `flights_geo` z zadania 8,
- `weather_clean` z zadania 6.

Dołącz pogodę po wspólnym kluczu:
- `origin`
- `year`
- `month`
- `day`
- `hour`

Po merge:
- użyj `validate='many_to_one'`,
- dodaj `indicator='merge_weather'`,
- zapisz wynik pod nazwą `analysis_df`.

Na końcu pokaż:
- `value_counts()` dla `merge_weather`,
- `value_counts(normalize=True)` dla `merge_weather`,
- kilka pierwszych wierszy z kolumnami pogodowymi.

Wymagane funkcje/parametry:
- `merge()` - połączenie z tabelą pogodową.
- `validate='many_to_one'` - kontrola relacji klucza.
- `indicator` i `value_counts(normalize=True)` - audyt dopasowania.

Kryterium zaliczenia:
- zadanie wykonane zgodnie z poleceniem,
- wynik pokazany w notebooku.

Checkpoint:
- pokaż prowadzącemu wynik końcowy tej sekcji.

Dokumentacja:
- https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html


In [135]:
# tutaj dodaj swój kod

# merge flights z weather
analysis_df = flights_geo.merge(
    weather_clean,
    on=["origin", "year", "month", "day", "hour"],
    how="left",
    validate="many_to_one",
    indicator="merge_weather"
)


print(analysis_df["merge_weather"].value_counts())
print(analysis_df["merge_weather"].value_counts(normalize=True))


# podgląd kolumn pogodowych
print(
    analysis_df[
        ["origin", "year", "month", "day", "hour", "temp", "wind_speed", "visib"]
    ].head()
)

merge_weather
both          324845
left_only       1524
right_only         0
Name: count, dtype: int64
merge_weather
both          0.99533
left_only     0.00467
right_only    0.00000
Name: proportion, dtype: float64
  origin  year  month  day  hour   temp  wind_speed  visib
0    EWR  2013      1    1     5  39.02    12.65858   10.0
1    LGA  2013      1    1     5  39.92    14.96014   10.0
2    JFK  2013      1    1     5  39.02    14.96014   10.0
3    JFK  2013      1    1     5  39.02    14.96014   10.0
4    LGA  2013      1    1     6  39.92    16.11092   10.0


## <span style="color:#0f766e; font-weight:700;">Zadanie 10</span>

Q1: Zrób ranking przewoźników po medianie `dep_delay`.

Pracuj na `analysis_df`.

Pogrupuj dane po:
- `carrier`
- `name`

Policz dokładnie te kolumny wynikowe:
- `flights_n`
- `dep_delay_mean`
- `dep_delay_median`
- `on_time_rate`
- `delay_recovery_mean`

Przyjmij:
- `on_time_rate = 1 - mean(is_delayed_dep)`

Zostaw tylko przewoźników z minimum `1000` lotów, posortuj rosnąco po `dep_delay_median` i pokaż najlepszych 15.

Wymagane funkcje/parametry:
- `groupby().agg()` - agregacja po przewoźniku.
- `query()` - filtr minimalnej liczebności.
- `sort_values()` - ranking wyników.

Kryterium zaliczenia:
- zadanie wykonane zgodnie z poleceniem,
- wynik pokazany w notebooku.

Checkpoint:
- pokaż prowadzącemu wynik końcowy tej sekcji.

Dokumentacja:
- https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html
- https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.query.html


In [ ]:
# tutaj dodaj swój kod


## <span style="color:#0f766e; font-weight:700;">Zadanie 11</span>

Q2: Wyznacz TOP 10 kierunków z najwyższym średnim `arr_delay`.

Pracuj na `analysis_df`.

Pogrupuj dane po:
- `dest`
- `dest_airport_name`

Policz dokładnie te kolumny wynikowe:
- `flights_n`
- `arr_delay_mean`
- `arr_delay_median`
- `distance_mean`
- `air_time_mean`

Zostaw tylko kierunki z minimum `500` lotów, posortuj malejąco po `arr_delay_mean` i pokaż TOP 10.

Wymagane funkcje/parametry:
- `groupby().agg()` - agregacja po kierunku.
- `query()` - filtr minimalnej liczebności.
- `head(10)` - ranking końcowy.

Kryterium zaliczenia:
- zadanie wykonane zgodnie z poleceniem,
- wynik pokazany w notebooku.

Checkpoint:
- pokaż prowadzącemu wynik końcowy tej sekcji.

Dokumentacja:
- https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.head.html


In [ ]:
# tutaj dodaj swój kod


## <span style="color:#0f766e; font-weight:700;">Zadanie 12</span>

Q3: Oceń relację między opóźnieniami a pogodą.

Pracuj na `analysis_df`.

Wykonaj dwa kroki:
1. Zbuduj podzbiór bez braków dla kolumn:
   - `dep_delay`
   - `temp`
   - `wind_speed`
   - `precip`
   - `visib`
   - `pressure`
2. Na tym podzbiorze:
   - policz korelacje,
   - podziel `pressure` na 4 przedziały przez `pd.qcut()` albo `pd.cut()`,
   - policz w tych grupach:
     - `flights_n`
     - `dep_delay_mean`
     - `dep_delay_median`

Zapisz tabelę korelacji jako `q3_corr`, a tabelę pasm ciśnienia jako `q3_pressure_groups`.

Wymagane funkcje/parametry:
- `corr()` - korelacje zmiennych liczbowych.
- `pd.qcut()` albo `pd.cut()` - przedziały ciśnienia.
- `groupby().agg()` - podsumowanie grup pogodowych.

Kryterium zaliczenia:
- zadanie wykonane zgodnie z poleceniem,
- wynik pokazany w notebooku.

Checkpoint:
- pokaż prowadzącemu wynik końcowy tej sekcji.

Dokumentacja:
- https://pandas.pydata.org/docs/reference/api/pandas.qcut.html
- https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html


In [ ]:
# tutaj dodaj swój kod


## <span style="color:#0f766e; font-weight:700;">Zadanie 13</span>

Q4: Sprawdź punktualność w czasie.

Pracuj na `analysis_df`.

Przygotuj trzy wyniki:
1. `q4_month` przez grupowanie po `month_num`
2. `q4_weekday` przez grupowanie po `weekday`
3. `q4_hour_pivot` jako tabelę `weekday x dep_hour`

W `q4_month` i `q4_weekday` policz:
- `flights_n`
- `on_time_rate`
- `dep_delay_median`

Przyjmij:
- `on_time_rate = 1 - mean(is_delayed_dep)`

W `q4_hour_pivot` użyj:
- `index='weekday'`
- `columns='dep_hour'`
- `values='is_delayed_dep'`
- `aggfunc='mean'`

Wymagane funkcje/parametry:
- `groupby().agg()` - podsumowania miesięczne i tygodniowe.
- `pivot_table()` - tabela do dalszej heatmapy.
- `sort_index()` albo `sort_values()` - uporządkowanie wyników.

Kryterium zaliczenia:
- zadanie wykonane zgodnie z poleceniem,
- wynik pokazany w notebooku.

Checkpoint:
- pokaż prowadzącemu wynik końcowy tej sekcji.

Dokumentacja:
- https://pandas.pydata.org/docs/reference/api/pandas.pivot_table.html


In [ ]:
# tutaj dodaj swój kod


## <span style="color:#0f766e; font-weight:700;">Zadanie 14</span>

Przygotuj dwa przekrojowe podsumowania: `pivot_table()` i `crosstab()`.

Pracuj na `analysis_df`.

Zrób dokładnie:
1. `q14_pivot` ze średnim `dep_delay` w układzie:
   - wiersze: `origin`
   - kolumny: `carrier`
   - wartości: `dep_delay`
2. `q14_crosstab` z udziałem klas `dep_delay_class`:
   - wiersze: `origin`
   - kolumny: `dep_delay_class`
   - normalizacja: `normalize='index'`

Na końcu pokaż obie tabele.

Wymagane funkcje/parametry:
- `pivot_table()` - tabela przestawna z agregacją.
- `pd.crosstab(..., normalize='index')` - udział klas w wierszach.

Kryterium zaliczenia:
- zadanie wykonane zgodnie z poleceniem,
- wynik pokazany w notebooku.

Checkpoint:
- pokaż prowadzącemu wynik końcowy tej sekcji.

Dokumentacja:
- https://pandas.pydata.org/docs/reference/api/pandas.pivot_table.html
- https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html


In [ ]:
# tutaj dodaj swój kod


## <span style="color:#0f766e; font-weight:700;">Zadanie 15</span>

Q5: Zbuduj końcowy audyt i mini-raport projektu.

Pracuj na wynikach z poprzednich zadań.

Twoja odpowiedź ma mieć 3 części:

1. Tabela audytu
   Zbuduj tabelę z krokami:
   - `flights raw`
   - `flights after_dropna`
   - `flights after_range_validation`
   - `weather raw`
   - `weather after_dedupe`
   - `analysis final_after_merges`

   Dla każdego kroku pokaż:
   - `dataset`
   - `step`
   - `rows`
   - `removed_vs_prev`

2. Trzy wykresy
   Zrób dokładnie trzy wykresy:
   - ranking przewoźników z Q1,
   - ranking kierunków z Q2,
   - `on_time_rate` wg miesiąca z Q4.

3. Krótkie wnioski tekstowe
   Napisz po 1 krótkiej odpowiedzi do:
   - Q1
   - Q2
   - Q3
   - Q4
   - Q5

Wymagane funkcje/parametry:
- `pd.DataFrame()` - tabela audytu.
- `matplotlib` i `seaborn` - trzy wykresy.
- krótkie wnioski tekstowe oparte na wcześniej policzonych wynikach.

Kryterium zaliczenia:
- zadanie wykonane zgodnie z poleceniem,
- wynik pokazany w notebooku.

Checkpoint:
- pokaż prowadzącemu wynik końcowy tej sekcji.

Dokumentacja:
- https://matplotlib.org/stable/api/pyplot_summary.html
- https://seaborn.pydata.org/


In [ ]:
# tutaj dodaj swój kod


# <span style="color:#0f766e; font-weight:700;">5. Zadania dodatkowe (16-24)</span>


## <span style="color:#0f766e; font-weight:700;">Zadanie 16 - dodatkowe</span>

Zbuduj heatmapę `weekday x dep_hour` dla `on_time_rate`.

Pracuj na `q4_hour_pivot` z zadania 13.

Pokaż heatmapę, w której:
- wiersze to `weekday`,
- kolumny to `dep_hour`,
- kolor pokazuje `on_time_rate`.

Jeśli Twoja tabela zawiera średnią z `is_delayed_dep`, pamiętaj, że:
- `on_time_rate = 1 - mean(is_delayed_dep)`.

Wymagane funkcje/parametry:
- `pivot_table()` - tabela wejściowa do heatmapy.
- `sns.heatmap()` - wizualizacja punktualności.

Kryterium zaliczenia:
- zadanie wykonane zgodnie z poleceniem,
- wynik pokazany w notebooku.

Checkpoint:
- pokaż prowadzącemu wynik końcowy tej sekcji.

Dokumentacja:
- https://seaborn.pydata.org/generated/seaborn.heatmap.html


In [ ]:
# tutaj dodaj swój kod


## <span style="color:#0f766e; font-weight:700;">Zadanie 17 - dodatkowe</span>

Przeanalizuj anulowane loty.

Pracuj na surowej tabeli `flights`.

Przyjmij definicję:
- lot anulowany = rekord z brakującym `dep_time`

Przygotuj dwa wyniki:
1. `cancel_by_month` z kolumnami:
   - `flights_n`
   - `cancel_rate`
2. `cancel_by_carrier` z kolumnami:
   - `flights_n`
   - `cancel_rate`

Dla przewoźników pokaż TOP 10 po najwyższym `cancel_rate`.

Wymagane funkcje/parametry:
- `isna()` - identyfikacja anulowanych lotów przez brak `dep_time`.
- `groupby().agg()` - agregacja po miesiącu i przewoźniku.
- `sort_values()` - ranking przewoźników.

Kryterium zaliczenia:
- zadanie wykonane zgodnie z poleceniem,
- wynik pokazany w notebooku.

Checkpoint:
- pokaż prowadzącemu wynik końcowy tej sekcji.

Dokumentacja:
- https://pandas.pydata.org/docs/reference/api/pandas.Series.isna.html


In [ ]:
# tutaj dodaj swój kod


## <span style="color:#0f766e; font-weight:700;">Zadanie 18 - dodatkowe</span>

Dołącz `planes` i oceń wpływ wieku samolotu na `arr_delay`.

Pracuj na `analysis_df`.

Wykonaj kolejno:
1. z `planes` wybierz kolumny:
   - `tailnum`
   - `year`
   - `manufacturer`
   - `model`
   - `seats`
2. zmień nazwę `year` na `plane_year`,
3. skonwertuj `plane_year` do typu liczbowego,
4. połącz dane z `analysis_df` po `tailnum`,
5. policz `plane_age = year - plane_year`,
6. zbuduj `plane_age_bin` przez `pd.cut()` z przedziałami:
   - `[0, 5, 10, 20, 40, 80]`
7. policz w tych grupach:
   - `flights_n`
   - `arr_delay_mean`

Wymagane funkcje/parametry:
- `merge()` - łączenie z tabelą planes.
- `pd.to_numeric()` - konwersja roku produkcji samolotu.
- `pd.cut()` - grupowanie wieku samolotu.

Kryterium zaliczenia:
- zadanie wykonane zgodnie z poleceniem,
- wynik pokazany w notebooku.

Checkpoint:
- pokaż prowadzącemu wynik końcowy tej sekcji.

Dokumentacja:
- https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html
- https://pandas.pydata.org/docs/reference/api/pandas.cut.html


In [ ]:
# tutaj dodaj swój kod


## <span style="color:#0f766e; font-weight:700;">Zadanie 19 - dodatkowe</span>

Znajdź najbardziej niestabilne trasy.

Pracuj na `analysis_df`.

Wykonaj kolejno:
1. utwórz `route = origin + '-' + dest`,
2. pogrupuj po `route`,
3. policz:
   - `flights_n`
   - `arr_delay_std`
   - `arr_delay_mean`
4. zostaw tylko trasy z minimum `200` lotów,
5. posortuj malejąco po `arr_delay_std`,
6. pokaż TOP 15.

Wymagane funkcje/parametry:
- `assign()` - utworzenie zmiennej `route`.
- `groupby().agg()` - liczebność i zmienność opóźnień.
- `query()` i `sort_values()` - ranking tras.

Kryterium zaliczenia:
- zadanie wykonane zgodnie z poleceniem,
- wynik pokazany w notebooku.

Checkpoint:
- pokaż prowadzącemu wynik końcowy tej sekcji.

Dokumentacja:
- https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.std.html


In [ ]:
# tutaj dodaj swój kod


## <span style="color:#0f766e; font-weight:700;">Zadanie 20 - dodatkowe</span>

Porównaj średnią, medianę i średnią obciętą `dep_delay` dla przewoźników.

Pracuj na `analysis_df`.

Wykonaj kolejno:
1. napisz własną funkcję `trimmed_mean()`,
2. dla każdego `carrier` policz:
   - `flights_n`
   - `dep_delay_mean`
   - `dep_delay_median`
   - `dep_delay_trimmed`
3. zostaw tylko przewoźników z minimum `1000` lotów,
4. posortuj wynik po `dep_delay_median`.

Jeśli chcesz, możesz przyjąć obcięcie 10% obserwacji z obu stron rozkładu.

Wymagane funkcje/parametry:
- własna funkcja `trimmed_mean()` - średnia obcięta.
- `groupby().agg()` - zestaw kilku miar naraz.
- `query()` i `sort_values()` - końcowa tabela rankingowa.

Kryterium zaliczenia:
- zadanie wykonane zgodnie z poleceniem,
- wynik pokazany w notebooku.

Checkpoint:
- pokaż prowadzącemu wynik końcowy tej sekcji.

Dokumentacja:
- https://docs.python.org/3/tutorial/controlflow.html#defining-functions


In [ ]:
# tutaj dodaj swój kod


## <span style="color:#0f766e; font-weight:700;">Zadanie 21 - dodatkowe</span>

Policz bootstrapowy 95% CI dla mediany `dep_delay` dla dwóch przewoźników.

Pracuj na `analysis_df`.

Wybierz dwóch największych przewoźników według liczby lotów. Następnie dla każdego z nich:
1. pobierz wektor `dep_delay`,
2. wykonaj bootstrap z losowaniem ze zwracaniem,
3. policz 1000 bootstrapowych median,
4. wyznacz:
   - medianę z danych,
   - `ci95_low`,
   - `ci95_high`

Końcowy wynik pokaż w tabeli z kolumnami:
- `carrier`
- `median_dep_delay`
- `ci95_low`
- `ci95_high`

Wymagane funkcje/parametry:
- `np.random.default_rng()` - losowanie bootstrapowe.
- `np.percentile()` - przedział ufności z rozkładu bootstrapowego.

Kryterium zaliczenia:
- zadanie wykonane zgodnie z poleceniem,
- wynik pokazany w notebooku.

Checkpoint:
- pokaż prowadzącemu wynik końcowy tej sekcji.

Dokumentacja:
- https://numpy.org/doc/stable/reference/generated/numpy.percentile.html


In [ ]:
# tutaj dodaj swój kod


## <span style="color:#0f766e; font-weight:700;">Zadanie 22 - dodatkowe</span>

Zbuduj prosty baseline `arr_delay ~ dep_delay + distance`.

Pracuj na `analysis_df`.

Wykonaj kolejno:
1. zbuduj podzbiór bez braków dla:
   - `arr_delay`
   - `dep_delay`
   - `distance`
2. policz macierz korelacji,
3. dopasuj prostą `arr_delay ~ dep_delay` przez `np.polyfit()`,
4. wypisz:
   - macierz korelacji,
   - `slope`,
   - `intercept`

Nie budujesz tu modelu ML, tylko bardzo prosty baseline opisowy.

Wymagane funkcje/parametry:
- `corr()` - korelacje między zmiennymi.
- `np.polyfit()` - prosta linia trendu.

Kryterium zaliczenia:
- zadanie wykonane zgodnie z poleceniem,
- wynik pokazany w notebooku.

Checkpoint:
- pokaż prowadzącemu wynik końcowy tej sekcji.

Dokumentacja:
- https://numpy.org/doc/stable/reference/generated/numpy.polyfit.html


In [ ]:
# tutaj dodaj swój kod


## <span style="color:#0f766e; font-weight:700;">Zadanie 23 - dodatkowe</span>

Wyeksportuj oczyszczoną tabelę i kluczowe wyniki.

Pracuj na:
- `analysis_df`
- `q1_carriers`
- `q2_destinations`

Wykonaj kolejno:
1. utwórz folder `dane_lekcja4_wyniki`,
2. zapisz do CSV:
   - `analysis_df`
   - `q1_carriers`
   - `q2_destinations`
3. przygotuj krótki słownik z metrykami:
   - `rows_final`
   - najlepszy przewoźnik
   - najgorszy kierunek
4. zapisz ten słownik do JSON.

Wymagane funkcje/parametry:
- `to_csv(index=False)` - eksport tabel do CSV.
- `json.dump(..., indent=2)` - eksport krótkiego raportu do JSON.

Kryterium zaliczenia:
- zadanie wykonane zgodnie z poleceniem,
- wynik pokazany w notebooku.

Checkpoint:
- pokaż prowadzącemu wynik końcowy tej sekcji.

Dokumentacja:
- https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html
- https://docs.python.org/3/library/json.html


In [ ]:
# tutaj dodaj swój kod


## <span style="color:#0f766e; font-weight:700;">Zadanie 24 - dodatkowe</span>

Napisz funkcję `run_lesson4_pipeline()` zwracającą słownik wyników Q1-Q5.

Twoja funkcja ma samodzielnie wykonać główny pipeline:
1. import danych,
2. przygotowanie `project_flights`,
3. `dropna()` i walidację zakresów,
4. budowę cech czasu i opóźnień,
5. przygotowanie `weather_clean`,
6. merge z `airlines`, `airports` i `weather`,
7. policzenie podstawowych wyników:
   - ranking przewoźników,
   - ranking kierunków,
   - korelacje pogodowe,
   - podsumowanie miesięczne,
   - audyt liczby usuniętych rekordów

Funkcja powinna zwrócić `dict` z kluczami:
- `analysis_df`
- `q1_carriers`
- `q2_destinations`
- `q3_corr`
- `q4_month`
- `q5_audit`

Wymagane funkcje/parametry:
- `def` - utworzenie funkcji.
- `dict` - zwrot wyników w uporządkowanej postaci.
- `dropna()`, `between()`, `merge()`, `groupby()` - główne kroki pipeline'u.

Kryterium zaliczenia:
- zadanie wykonane zgodnie z poleceniem,
- wynik pokazany w notebooku.

Checkpoint:
- pokaż prowadzącemu wynik końcowy tej sekcji.

Dokumentacja:
- https://docs.python.org/3/tutorial/controlflow.html#defining-functions


In [ ]:
# tutaj dodaj swój kod


## <span style="color:#0f766e; font-weight:700;">Najczęstsze błędy</span>


- łączenie `weather` bez wcześniejszego usunięcia duplikatów po kluczu,
- porównywanie przewoźników bez minimalnej liczebności lotów,
- interpretacja średnich opóźnień bez mediany i liczby obserwacji,
- brak audytu: ile rekordów odpadło na `dropna`, ile na walidacji zakresów, ile przy merge,
- traktowanie anulowanych lotów i braków tak, jakby były zwykłymi obserwacjami,
- budowanie wykresów bez wcześniejszego sprawdzenia, jakie rekordy weszły do analizy.
